# 프로젝트 1 - Weekend 2: RAG 파이프라인

| 항목 | 내용 |
|------|------|
| **프로젝트** | 주택청약 FAQ 챗봇 - 벡터 검색 업그레이드 |

FAQ 챗봇에 벡터 검색을 업그레이드 해주세요

## 환경 설정

In [ ]:
# !pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv gradio

In [44]:
# colab 사용 시 아래 주석 해제
!pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv gradio
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [4]:
# 환경 설정
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS

client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ 환경 설정 완료")

✅ 환경 설정 완료


## 📦 실습용 샘플 데이터

In [5]:
# ============================================================
# 주택청약 FAQ 샘플 데이터 (실습용)
# ============================================================
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장",
     "question": "주택청약종합저축이란 무엇인가요?",
     "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨",
     "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
    {"id": "FAQ004", "category": "청약통장",
     "question": "청약통장 1순위 조건은 무엇인가요?",
     "answer": "1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입",
     "keywords": ["1순위", "가입기간", "예치금", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ005", "category": "청약자격",
     "question": "주택 청약 신청 자격 조건은 무엇인가요?",
     "answer": "1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능",
     "keywords": ["청약자격", "만19세", "무주택", "세대주"], "difficulty": "easy"},
    {"id": "FAQ006", "category": "청약자격",
     "question": "무주택자 기준은 무엇인가요?",
     "answer": "본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함",
     "keywords": ["무주택", "세대원", "소형주택", "분양권"], "difficulty": "medium"},
    {"id": "FAQ009", "category": "특별공급",
     "question": "특별공급의 종류에는 어떤 것이 있나요?",
     "answer": "1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대",
     "keywords": ["특별공급", "기관추천", "다자녀", "신혼부부", "생애최초"], "difficulty": "medium"},
    {"id": "FAQ010", "category": "특별공급",
     "question": "신혼부부 특별공급 조건은 무엇인가요?",
     "answer": "1) 혼인기간 7년 이내 무주택 세대주\n2) 소득: 도시근로자 월평균소득 100~140%\n3) 전용면적 85㎡ 이하\n4) 혼인기간 짧을수록 + 자녀 많을수록 가점 높음\n5) 예비 신혼부부도 신청 가능",
     "keywords": ["신혼부부", "혼인기간", "소득기준", "가점"], "difficulty": "medium"},
    {"id": "FAQ013", "category": "일반공급",
     "question": "가점제와 추첨제의 차이는 무엇인가요?",
     "answer": "가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%",
     "keywords": ["가점제", "추첨제", "84점", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ017", "category": "당첨/계약",
     "question": "당첨자 발표는 어떻게 확인하나요?",
     "answer": "1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인",
     "keywords": ["당첨자발표", "청약홈", "SMS알림", "서류제출"], "difficulty": "easy"},
    {"id": "FAQ020", "category": "당첨/계약",
     "question": "재당첨 제한이란 무엇인가요?",
     "answer": "당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)",
     "keywords": ["재당첨제한", "10년", "7년", "세대원"], "difficulty": "medium"},
    {"id": "FAQ023", "category": "기타",
     "question": "청약홈 사이트는 어떻게 이용하나요?",
     "answer": "청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공",
     "keywords": ["청약홈", "공인인증서", "간편인증", "가점계산"], "difficulty": "easy"},
]

SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
]

print(f"📦 FAQ 데이터 로드 완료: {len(SAMPLE_FAQ_DATA)}개 QA, {len(SAMPLE_TEST_QUERIES)}개 테스트 질의")

📦 FAQ 데이터 로드 완료: 10개 QA, 5개 테스트 질의


---
## 사이클 1: Weekend 1 복원 + Document 객체

FAQ 데이터를 LangChain `Document` 객체 리스트로 변환하세요. `page_content`에 질문+답변, `metadata`에 id/category를 넣으세요.

---

### ⭐[내용정리] Document 클래스
- LangChain의 가장 기본적인 데이터 컨테이너 클래스. RAG 파이프라인에서 텍스트 조각과 그 출처/속성 정보를 함께 묶어 다루는 표준 단위임.
- Document는 내부적으로 Pydantic 모델 기반이며 세개의 필드로 구성
```Python
class Document(BaseMedia):
    page_content: str          # ✅ 필수 — 실제 텍스트 내용
    metadata:     dict = {}    # 선택 — 출처, 카테고리 등 부가 정보
    id:           str | None   # 선택 — 문서 고유 식별자 (UUID 권장)
    type:         "Document"   # 내부 고정값, 직접 지정 불필요
```
1. 필드별 상세 설명
  - `page_content`(필수): 실제 검색·임베딩에서 사용되는 핵심 텍스트임. RAG시, FAISS 또는 retriever는 해당 필드만 임베딩.
    - 예시
      ```Python
      # 1) 아무 인자값 명시 없이, 문자열만 줄 수 있음.
      doc = Document("안녕하세요, LangChain입니다.")
      # 2) 명시적으로 인자에 값을 넣을 수 있음.
      doc = Document(page_content="안녕하세요, LangChain입니다.")

      # >>> 출력 예시
      print(doc.page_content)
      # >>> "안녕하세요, Langchain입니다."
      ```
  - `metadata`(`{}`):문서의 출처, 카테고리, 필터링 기준 등 속성정보를 담는 딕셔너리임. 어떤 `키-값` 쌍이든 자유롭게 넣을 수 있음.
    - 예시
      ```Python
        doc = Document(
            page_content="재택근무는 주 2회까지 가능하다.",

            metadata ={     
              "source": "company_policy.txt", # 파일출처
              "category": "사내규정", # 분류
              "page": 3 # 페이지 번호
              "author": "HR팀", # 작성자
              "created": "2024-01-01" # 날짜
            }
        )
      # >>> 출력 예시
      print(doc.metadata)
      # >>> {'source': 'company_policy.txt', 'category': '사내규정', 'page':3, ...}

      # 개별 필드 접근 (딕셔너리의 get() 사용)
      print(doc.metadata.get("category"))
      # >>> "사내규정"
      ```
  - `id`(기본값 `None`): 특정 문서를 고유하게 식별할 때 사용. FAISS 등 벡터스토어가 내부적으로 자동 부여할 수도 있음.
    - 예시
      ```Python
        doc = Document(
                page_content="텍스트",
                id = str(uuid.uuid4()) # "bb87d1d2-e75b-499e-9669-53f8ee7b28c6"
        )
      ```

2. 객체 출력 구성
  - 예시
    ```python
    doc = Document(
            page_content="파이썬은 데이터 분석에 좋다",
            metadata={"source": "intro.txt", "category": "기술문서"}
        )
    
    # >>> 출력예시
    print(doc)
    # >>> page_content='파이썬은 데이터 분석에 좋다' metadata={'source': 'intro.txt', 'category': '기술문서'}

    repr(doc)
    # >>> Document(page_content='파이썬은 데이터 분석에 좋다', metadata={'source': ...}, id=None

    ```
  - `Document`는 딕셔너리가 아닌, **Pydantic 객체**임.
  - (*추가) Pydantic
    - Pydantic은 Python 타입 힌트를 기반으로 데이터를 검증하고, 구조화된 객체로 변환해주는 라이브러리.
    - LangChain의 대부분의 클래스(`Document`, `ChatOpenAI`등)가 Pydantic 기반으로 만들어져 있음.
    - 주요 기능은 형식을 검사하는 것: 일반 Python 클래스는 어떤 타입이든 그냥 받아들이지만, Pydantic은 입력 데이터가 약속된 형식에 맞는지 자동으로 검사
    - 예시
      ```text
        외부 데이터 (dict, JSON, str...)
              ↓
        Pydantic Model (검문소)
        ✅ 타입 검증
        ✅ 자동 형변환
        ✅ 기본값 처리
              ↓
        안전한 Python 객체
      ```




In [73]:
# 사이클 1: FAQ 데이터를 LangChain Document 객체 리스트로 변환.
  # page_content에 질문+답변, metadata에 id/category를 넣으세요.
"""
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장",
     "question": "주택청약종합저축이란 무엇인가요?",
     "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨",
     "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
"""

from langchain_core.documents import Document

# [계획]
  # for루프를 돌면서 for dat in SAMPLE_FAQ_DATA
  # q = dat.get("question"), a = dat.get("answer") 한 뒤, join..? 아니면 content = q + a
  # id = dat.get('id')
  # cat = dat.get('category')
# Document 객체는 `page_content`,`metadata`, `id` 의 주요 세개의 속성을 가짐
# 1. 근데 어떻게 집어넣지?
  # (Test1) 샘플 하나만 넣어서 Document 객체로 만들어봄
# 2. 여러개는 어떻게 넣지?
  # 1) 하나의 Document 객체를 list처럼 고려해서 .append()처럼 계속 붙이는 방식?
  # 2) 전체 문서를 쪼개어 리스트처럼 만든 후 한번에 Document 객체에 넣는 방식?
    # 예를 들면 content = ["string1", "string2", ...], id = ['id1', 'id2',...]
  # LLM 답변은, `List[Document]` 패턴이라고 함.
    # Document 하나는 항상 단일 텍스트 조각 하나만 담고,
    # 여러 개를 처리할 때는 Python 리스트에 .append()로 쌓아나감.
    # `FAISS.from_documents()`도 인자로 list[Document]를 받도록 설계.
# (구조 예시)

"""
  docs = [
      Document(id='FAQ001', metadata={...}, page_content='Q: ...'),  ← 원소 0
      Document(id='FAQ004', metadata={...}, page_content='Q: ...'),  ← 원소 1
      Document(id='FAQ007', metadata={...}, page_content='Q: ...'),  ← 원소 2
      ...
  ]

  # 사용시 docs = []로 초기화,
    # 루프마다 Document 하나씩 추가
  docs.append(
          Document(
              page_content=content,
              id=id,
              metadata={"category": cat}
          )
      )
"""

# (Test1) -----------------------
# sample_of_sample = SAMPLE_FAQ_DATA[0]
# q = f"Q: {sample_of_sample.get("question")}\n"
# a = f"A: {sample_of_sample.get("answer")}"
# content = q +  a
# print(f"sample content: {content}")

# id = sample_of_sample.get('id')
# print(f"sample id: {id}")

# cat = sample_of_sample.get('category')
# print(f"sample category: {cat}")

# docs = Document(
#     page_content = content,
#     id = id,
#     metadata = {
#        "category": cat
#     }
#     )
#(/Test1) -------------------------

docs = []

for sample in SAMPLE_FAQ_DATA:
  q = f"Q: {sample.get("question")}\n"
  a = f"A: {sample.get("answer")}"
  content = q + a
  id = sample.get('id')
  cat = sample.get('category')

  docs.append(Document(
      page_content = content,
      id = id,
      metadata = {
          "category":cat
      }
  )
  )


In [30]:
len(docs), len(SAMPLE_FAQ_DATA)

(10, 10)

In [31]:
docs

[Document(id='FAQ001', metadata={'category': '청약통장'}, page_content='Q: 주택청약종합저축이란 무엇인가요?\nA: 주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨'),
 Document(id='FAQ004', metadata={'category': '청약통장'}, page_content='Q: 청약통장 1순위 조건은 무엇인가요?\nA: 1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입'),
 Document(id='FAQ005', metadata={'category': '청약자격'}, page_content='Q: 주택 청약 신청 자격 조건은 무엇인가요?\nA: 1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능'),
 Document(id='FAQ006', metadata={'category': '청약자격'}, page_content='Q: 무주택자 기준은 무엇인가요?\nA: 본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함'),
 Document(id='FAQ009', metadata={'category': '특별공급'}, page_content='Q: 특별공급의 종류에는 어떤 것이 있나요?\nA: 1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년

In [36]:
docs[0].page_content

'Q: 주택청약종합저축이란 무엇인가요?\nA: 주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨'

---
## 사이클 2: OpenAI Embeddings

`OpenAIEmbeddings`로 FAQ 텍스트들을 임베딩하고, 질문 간 코사인 유사도를 계산하세요. 의미적으로 유사한 질문이 높은 유사도를 보이는지 확인하세요.


---
### ⭐ [내용정리]

#### 1. OpenAIEmbeddings
- from langchain_openai import OpenAIEmbeddings
  - OpenAIEmbeddings: 텍스트를 숫자 벡터로 변환해주는 API 래퍼. `FAISS.from_document()` 내부에서도 해당 클래스가 자동으로 호출됨
- 초기화 주요 인자
  ```python
  from langchain_openai import OpenAIEmbeddings

  embeddings = OpenAiEmbeddings(
    model = "text-embedding-3-small" # 사용할 임베딩 모델명
    api_key = "sk-..." # 생략 시 api키 자동 참조 (OPENAI_API_KEY)
    dimensions = 512 # 출력 벡터 차원 수 축소 가능(원래는 해당 모델은 1536차원)
    chunk_size = 1000, # 한 번에 API에 보낼 텍스트 묶음 크기
  )
  ```
- 핵심 메서드(embeddings)
  1. `embed_query(text)`: 단일 쿼리용
  ```python
  # 인자: str
  # 반환: list[float] (벡터 1개)
  emb = embeddings.embed_query("주택청약이란 무엇인가요?")

  print(type(emb))
  # >>> <class 'list'>
  print(len(emb))
  # >>> 1536 (text-embedding-3-small 기준)
  print(emb[:3])
  # >>> [0.0192, -0.0645, -0.0016, ...]
  ```
  2. `embed_documents(text)`: 여러 문서용
  ```python
  # 인자: list[str] (문자열 리스트)
  # 반환: list[list[float]] (벡터 리스트)

  texts = [
    "주택청약종합저축은 만능 통장입니다.",
    "1순위 조건은 12개월 이상 납입입니다.",
    "재택근무는 주 2회까지 가능합니다."
  ]
  embs = embeddings.embed_documents(texts)

  print(type(embs))         
  # >>> <class 'list'>
  print(len(embs))          
  # >>> 3 (입력 문장 개수)
  print(len(embs[0]))       
  # >>> 1536 (각 벡터의 차원)
  print(type(embs[0]))      
  # >>> <class 'list'>
  # 즉, vecs = [[float, float, ...], [float, float, ...], ...]
  ```
  3. 차이점
  ```text
  embed_query("텍스트")       →  [0.01, -0.06, ...]          # 1D 리스트
  embed_documents(["a","b"]) →  [[0.01,...], [0.03,...]]     # 2D 리스트 (리스트의 리스트)

  ```

#### 2. `cosine_similarity` (sklearn)
- 두 벡터가 얼마나 같은 방향을 가리키는지를 -1 ~ 1 사이 값으로 나타냄.
  ```
  K(X,Y)= ⟨X,Y⟩ /(∥X∥⋅∥Y∥)
  ```
- 실무에서는 사용하지 않음
  - 전수 탐색: 문서가 1만 개라면 쿼리마다 1만 번 코사인 유사도를 계산함. 문서가 100만 개, 1억 개가 되면 응답 속도가 수십 초~분 단위로 늘어나서 실용적이지 않음.
  - FAISS의 경우 클러스터링, 계층적 그래프탐색, 양자화 등등으로 검색 속도를 높임.
  - 관련 벡터 DB
    - 프로토타입/소규모: LangChain + FAISS
    - 중규모 서비스: Chroma, Qdrant (서버 방식 벡터 DB)
    - 대규모 엔터프라이즈: Pinecone, Weaviate, pgvector (관리형/분산형)

- 함수 시그니처 및 인자
  ```python
  from sklearn.metrics.pairwise import cosine_similarity

  cosine_similarity(X, Y=None, dense_output=True)
  ```
  | 인자           | 타입                                          | 설명                           |
  | ------------ | ------------------------------------------- | ---------------------------- |
  | X            | array-like, shape (n_samples_X, n_features) | 2D 배열 필수 — 행 하나가 벡터 하나       |
  | Y            | array-like, shape (n_samples_Y, n_features) | 비교 대상. None이면 X끼리 비교         |
  | dense_output | bool                                        | 기본 True, 희소행렬 입력 시 밀집 행렬로 출력 |

- 🚨 주의점: 반드시 2D 배열이어야 함.
  - query 임베딩을 [] 로 감싸서 2D로 만들기
  ```python
    # ❌ 잘못된 사용 — 1D 벡터 직접 입력
    q_vec = embeddings.embed_query("청약이란?")  # shape: (1536,)
    cosine_similarity(q_vec, doc_vecs)           # → 오류 또는 잘못된 결과

    # ✅ 올바른 사용 — [] 로 감싸서 2D로 만들기
    cosine_similarity([q_vec], doc_vecs)         # shape: (1, 1536) vs (N, 1536)
  ```

- 반환값 구조
  ```python
    q_vec   = embeddings.embed_query("청약통장 1순위")
    doc_vecs = embeddings.embed_documents([
        "청약 1순위는 12개월 이상 납입입니다.",   # 문서 0
        "재택근무는 주 2회 가능합니다.",           # 문서 1
        "파이썬은 데이터 분석에 좋습니다."         # 문서 2
    ])

    scores = cosine_similarity([q_vec], doc_vecs)

    print(type(scores))    
    # >>> <class 'numpy.ndarray'>
    print(scores.shape)    
    # >>> (1, 3): (쿼리 수, 문서 수)
    print(scores)
    # >>> array([[0.8712, 0.1023, 0.0541]])
    #           ↑문서0   ↑문서1   ↑문서2

    # 쿼리가 1개인 경우 [0]으로 1D로 꺼냄
    scores = cosine_similarity([q_vec], doc_vecs)[0]
    print(scores)          
    #>>> [0.8712, 0.1023, 0.0541]
    print(scores.shape)    
    # >>> (3,)
  ```

In [63]:
# 사이클 2: OpenAIEmbeddings로 FAQ 텍스트들을 임베딩하고,
  # 질문 간 코사인 유사도를 계산하세요.
  # 의미적으로 유사한 질문이 높은 유사도를 보이는지 확인하세요
import numpy as np
from langchain_openai import OpenAIEmbeddings

# [계획]
  # FAQ 텍스트를 임베딩해야함.
    # embeddings(OpenAIEmbeddings)의 두가지 메서드
      # embed_query -> 결과는 리스트(벡터 하나)
      # embed_documents -> 결과는 2차원 리스트(벡터들의 리스트)
    # 우선 texts = []로 빈 리스트 초기화 해놓고
    # for 루프를 이용해서 앞서 만들어놓은 docs(Document 객체)의 속성
      # for doc in docs:
        # text = doc.page_content로 문자열 추출
        # texts.append(text)로 문장들 수집
    # 수집된 texts 리스트 전체를 embeddings.embed_documents(texts)에 입력
      # 이러면 2차원 리스트형태, 각 요소는 embs vector임.
  # FAQ 텍스트-질문간 코사인 유사도를 계산해야함
    # query는 하나만 뽑아서 embeddings.embed_query(query) 사용
    # 비교는 코사인 유사도로, cosine_similarity([emb_query], embs_texts)

"""
SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
]
"""
from sklearn.metrics.pairwise import cosine_similarity

embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

texts = []
for doc in docs:
  text = doc.page_content
  texts.append(text)
print(f"수집된 텍스트들(texts 변수) 개수: {len(texts)}")

# 임베딩 수행
emb_texts = embeddings.embed_documents(texts)
sample_query = SAMPLE_TEST_QUERIES[0].get("query")
emb_query = embeddings.embed_query(sample_query)

# 코사인 유사도 계산
score = cosine_similarity([emb_query], emb_texts)



수집된 텍스트들(texts 변수) 개수: 10


In [67]:
# score[0]을 사용하여 1차원 배열로 변환 후 텍스트와 결합
# 유사도(score) 기준 내림차순 정렬
sorted_results = sorted(zip(texts, score[0]), key=lambda x: x[1], reverse=True)

print(f"질문: {sample_query}\n")
print("--- 검색 결과 (유사도 순) ---")
for i, (text, s) in enumerate(sorted_results[:3]): # 상위 3개 출력
    print(f"[{i+1}] 유사도: {s:.4f}")
    print(f"{text}")
    print()

질문: 청약통장 가입하려면 어떻게 해요?

--- 검색 결과 (유사도 순) ---
[1] 유사도: 0.5032
Q: 청약홈 사이트는 어떻게 이용하나요?
A: 청약홈(www.applyhome.co.kr) - 한국부동산원 운영
1) 회원가입 후 공인인증서/간편인증 로그인
2) 청약 신청, 당첨 확인, 가점 계산 가능
3) 모바일 앱(청약홈)도 동일 서비스 제공

[2] 유사도: 0.4995
Q: 당첨자 발표는 어떻게 확인하나요?
A: 1) 청약홈(www.applyhome.co.kr) 접속
2) 당첨자 조회 메뉴 클릭
3) 문자 알림 서비스 신청 가능
※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인

[3] 유사도: 0.4021
Q: 주택청약종합저축이란 무엇인가요?
A: 주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.
1) 매월 2만원~50만원 자유 납입
2) 가입 후 일정 기간 경과 시 청약 자격 부여
3) 2009년 5월 이후 모든 청약통장이 통합됨



In [47]:
sorted(score)

[array([0.40194935, 0.31056483, 0.39216162, 0.07182064, 0.2531478 ,
        0.17757288, 0.1715686 , 0.49952461, 0.225679  , 0.50219139])]

In [38]:
len(texts)

10

In [34]:
docs[0].page_content

'Q: 주택청약종합저축이란 무엇인가요?\nA: 주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨'

---
## 사이클 3: FAISS 벡터 스토어

`FAISS.from_documents()`로 벡터 스토어를 만들고, `similarity_search()`와 `similarity_search_with_score()`로 검색하세요. 저장/로드도 테스트하세요.

---
### ⭐[내용정리] 벡터스토어, FAISS

#### 1. 벡터스토어란?
- 기존 SQL DB와의 차이
 - SQL은 "정확히 일치하는것(match)"을 찾고, 벡터스토어는 "의미가 비슷한 것(semantic similarity)"을 찾음.
 - 벡터스토어는 왜 '시간의 흐름'을 반영하지 못하는지?
  - 벡터는 임베딩 시점의 스냅샷이기 때문.
  - 임베딩은 한번 계산하면 고정된 숫자 배열로 저장됨. 시간적 맥락은 벡터 공간 상에서 표현이 불가능하며, 새로운 정보가 추가되려면 문서를 다시 임베딩하고 다시 색인해야함.
  - 따라서 실시간 정보가 필요한 경우 RAG만으로는 한계가 있으며, 웹 검색 도구(Tool)를 LLM에 붙이는 방식으로 보완함.

#### 2. FAISS 개요
- FAISS와 주요 벡터 DB 비교
  | 항목            | FAISS                 | Chroma               | Qdrant      | Pinecone              |
| ------------- | --------------------- | -------------------- | ----------- | --------------------- |
| 형태            | 라이브러리 (로컬)            | 임베디드 DB              | 독립 서버 DB    | 완전 관리형 클라우드           |
| 설치            | pip install faiss-cpu | pip install chromadb | Docker / 서버 | API 키만 발급             |
| 영속성           | 수동 저장 필요 (save_local) | 자동                   | 자동          | 자동 (클라우드)             |
| 메타데이터 필터링     | ⚠️ 제한적                | ✅ 가능                 | ✅ 강력        | ✅ 가능                  |
| 실시간 업데이트      | ⚠️ 불편                 | ✅ 가능                 | ✅ 가능        | ✅ 가능                  |
| 검색 속도 (1M 벡터) | ⚡ ~5-10ms (최고)        | ~20-50ms             | ~5-10ms     | ~15-30ms (네트워크 지연 포함) |
| GPU 가속        | ✅ 지원                  | ❌                    | ❌           | ❌                     |
| 비용            | 무료                    | 무료                   | 무료 (셀프호스팅)  | 유료                    |
| 추천 용도         | 프로토타입, 연구, 오프라인       | 로컬 개발                | 프로덕션        | 대규모 클라우드 서비스          |
  - 임베디드 DB: 특정 프로그램과 결합된 DB, 다른 컴퓨터나 여러 프로그램에서 동시에 접속해서 쓰기에는 적합하지 않음.
  - 독립 서버 DB: 프로그램과는 완전히 분리되어, 별도로 24시간 켜져 있는 백그라운드 서버 형태로 돌아가는 DB
   - MySQL, Oracle, PostgreSQL 같은 전통적인 DB 방식, 다양한 단말에서 접근가능.

- FAISS가 로컬에서만 쓰이는 이유
  - FAISS는 서버 기능이 없으며, 네트워크 API, 인증, 다중 사용자 동시 접속, 데이터 복제 같은 DB의 기본 기능이 없고 그냥 메모리 위의 인덱스 객체임. 프로그램이 꺼지면 save_local()로 저장했다가 load_local()로 다시 불러와야 함.
    - 예: FAISS = 엑셀 파일 (검색은 가능하나 서버/DB는 아님), Pinecone = 구글 스프레드시트 (클라우드에서 누구나 API로 접근가능)
  - Chroma, Qdrant, Pinecone 등은 FAISS가 못하는 서버·영속성·필터링 기능을 추가한 것.
    - 내부적으로 Chroma와 Qdrant도 인덱싱 알고리즘은 FAISS와 유사한 HNSW를 사용.
    - 교육과정 → FAISS, 실제 서비스 → Qdrant or Pinecone이 현업 흐름

#### 3. LangChain FAISS API 메서드 정리
'생성 메서드'와 '검색 메서드' 및 '저장/로딩 메서드'로 기능이 나뉨

1. ⚙️ 생성 메서드
  - `FAISS.from_documents()`: `Document` 객체 리스트를 받은 후 → 각 `page_content`를 임베딩 → FAISS 인덱스 생성 후 반환.
    - 예시
      ```python
      # 인자
        # documents: List[Document]
        # embedding: 임베딩 모델 객체 (OpenAIEmbeddings 등)
      # 반환: FAISS 객체 (vectorstore)

      vectorstore = FAISS.from_documents(docs, embeddings)
      ```
  - `FAISS.from_texts(texts, embedding, metadatas=None, ids=None)`: `Document` 객체 없이, 순수 문자열 리스트로 바로 인덱스를 만들 때 사용.
    - 예시
      ```python
        # from_documents와 기능은 같으나, 입력은 str 리스트 (List[str]

        texts=["청약통장이란...", "1순위 조건은..."]
        metas = [{"category": "청약"}, {"category": "청약"}]

        vectorstore = FAISS.from_texts(texts, embeddings, metadatas=metas)
      ```

2. 🔍 검색 메서드: search와 search with score가 있음.또한 문자열 대신 vector로 직접 검색하는 메서드도 존재

  - 1) `similarity_search(query, k=4, filter=None)`: 쿼리(query) 문자열과 가장 유사한 문서 k개를 `Document` 리스트로 반환.
    - 예시
      ```python
      # 인자
        # query: str    -- 검색할 질문
        # k: int        -- 반환할 문서 수 (기본 4)
        # filter: dict  -- metadata 기준 필터 (예: {"category":"청약통장"}
      # 반환: List[Document]

      results = vectorsotre.similarity_search("청약 1순위 조건", k=3)

      for doc in results:
        print(doc.page_content) # 문서 내용
        print(doc.metadata['category']) # 메타데이터
      ```
  - 2) `similarity_search_with_score(query, k=4)`: 문서와 함께 유사도 점수도 반환. ⚠️점수가 낮을수록 유사함 (L2 거리 기반).
    - 예시
      ```python
      # 입력: query: str
      # 반환: List[Tuple[Document, float]] -- 문서와 함께 유사도 점수를 반환
      # L2 거리는 낮을수록 유사
      results = vectorstore.similarity_search_with_score("청약 1순위 조건", k=3)

      for doc, score in results:
        print(f"거리:{score:.4f} | {doc.page_content[:40]}")

      # >>> 거리: 0.2341 | Q: 청약통장 1순위 조건은...  ← 가장 유사
      # >>> 거리: 0.6712 | Q: 주택청약종합저축이란...
      # >>> 거리: 1.2034 | Q: 재택근무는 주 2회...      ← 가장 거리 멀음
      ```
      
  - 3) (추가) `similarity_search_by_vector(embedding, k=4)`: 쿼리 문자열 대신 이미 만들어진 벡터로 검색
    - 예시
      ```python
      # embed_query로 먼저 벡터 생성
      q_vec = embeddings.embed_query("청약 1순위 조건")

      # 벡터로 직접 검색 (API 비용 절약)
      results = vectorstore.similarity_search_by_vector(q_vec, k=3)
      ```
  - 4) (추가) `max_marginal_relevance_search(query, k=4, fetch_k=20, lambda_mult=0.5)`: 유사도와 다양성을 동시에 고려한 k개를 반환, 검색 결과가 비슷한 문서들만 몰리는 문제(중복)를 방지.

    - 예시
      ```python
        # lambda_mult: 0이면 다양성 최대, 1이면 유사도 최대 (기본 0.5)
        # fetch_k: 먼저 가져올 후보 수 (여기서 다양성 필터링)

        results = vectorstore.max_marginal_relevance_search(
            "청약통장 조건",
            k=3,           # 최종 반환 개수
            fetch_k=10,    # 후보 10개 중 다양성 고려해서 3개 선택
            lambda_mult=0.5
          )
      ```
      
3. 💾 저장/로드 메서드
  - 인덱스를 로컬 파일 저장 및 로딩: `save_local(folder_path)` / `FAISS.load_local(folder_path, embeddings, allow_dangerous_deserialization=True)`

  - 예시
    ```python
    # 저장 → .faiss 파일과 .pkl 파일 생성
    vectorstore.save_local("faiss_index")

    # 불러오기 (보안 경고 옵션 명시 필요)
    loaded = FAISS.load_local(
        "faiss_index",
        embeddings,
        allow_dangerous_deserialization=True  # pkl 역직렬화 허용
    )
    print(loaded.index.ntotal)  # 저장된 벡터 수 확인
    ```

4. ➕📄 문서 추가 메서드
  - `add_documents(documents)` / `add_texts(texts, metadatas=None)`: 이미 만들어진 vectorstore에 문서를 추가함. 점진적으로 vectorstore를 확장할 때 사용.
  - 예시
    ```python
    new_docs = [Document(page_content="새 FAQ 내용", metadata = {"category": "신규"})]

    vectorstore.add_documents(new_docs)

    print(vectorstore.index.ntotal) # 추가 후 총 벡터 수 확인
    ```

5.⛓️LangChain Retriever 객체로 변환(LCEL 체인 연결용)
  - `as_retriever()`: vectorstore를 LangChain Retriever 객체로 변환. `|` 파이프 체인에 연결할 수 있음.
  - 예시
    ```python
    retriever = vectortore.as_retriever(
      search_type="smilarity", # "similarity" | "mmr" | "similarity_score_threshold"
      search_kwargs = {
        "k":3, # 반환 개수
        "filter": {"category": "청약통장"}, # 메타데이터 필터
        "score_threshold": 0.5 # similarity_score_threshold 사용 시
      }
    )

    # LCEL 체인에 바로 연결 가능
    rag_chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | ...
    ```



In [102]:
# 사이클 3
# 1) FAISS.from_documents()로 벡터 스토어를 만들고,
# 2) similarity_search()와 similarity_search_with_score()로 검색하세요.
# 3) 저장/로드도 테스트하세요.
from langchain_community.vectorstores import FAISS

# [계획]
# 1) FAISS.from_documents()로 벡터 스토어 만들기.
  # FAISS.from_documents()의 경우
  # Document 객체 리스트를 받은 후 → 각 page_content를 임베딩 → FAISS 인덱스 생성 후 반환.
  # DOcument 객체 리스트의 경우 `사이클 1`에서 만들어 놓은 docs 객체가 있음

# 2) similarity_search()와 similarity_search_with_score()로 검색.
  # vectorstore 의 메서드임. 쿼리(query)를 입력하여 유사한 문서를 검색

# 3) 저장/로드 테스트
  # 저장: vectorstore.save_local("faiss_index"), 입력한 문자열로 파일 생성?
  # 로드: loaded = FAISS.load_local(
  #                  "faiss_index",
  #                   embeddings,
  #                   allow_dangerous_deserialization=True  # pkl 역직렬화 허용
  #     )
  # 저장한 파일명, 임베딩, 기타..

# ==========================================================================================

# 1)
# Document 객체의 리스트인 docs를 넣기, 대신 embedding 임베딩 모델 객체도 함께 인자로 줘야함.
# 인자
  # documents: List[Document]
  # embedding: 임베딩 모델 객체 (OpenAIEmbeddings 등)
# 반환: FAISS 객체 (vectorstore)

vectorstore = FAISS.from_documents(docs, embeddings)


# 2)
# 2-1) 단순 search
query = "당첨되면 어떻게 확인해요?"
results = vectorstore.similarity_search(query)
print("2-1\n")
print(f"k개 중 첫번째 답변 확인:\n {results[0].page_content}")
print("="*50)

# 2-2) 점수 search
results_with_score = vectorstore.similarity_search_with_score(query, k=3)
print("2-2\n")
for re, sc in results_with_score:
  print(f"- 답변:\n{re.page_content},\n- 카테고리: {re.metadata.get('category')} \n-점수: {sc} \n")

# 3)
# vectorstore.save_local("faiss_index_2026-03-21")
# 폴더 "faiss_index_2026-03-21"에 index.faiss 및 pkl 파일 확인

loaded_vectorstore = vectorstore.load_local(
    "faiss_index_2026-03-21",
     embeddings,
     allow_dangerous_deserialization=True  # pkl 역직렬화 허용
)

print("="*50)
print("Loaded 벡터스토어 이용")
result_loaded = loaded_vectorstore.similarity_search(query)
print("2-1\n")
print(f"k개 중 첫번째 답변 확인:\n {result_loaded[0].page_content}")


2-1

k개 중 첫번째 답변 확인:
 Q: 당첨자 발표는 어떻게 확인하나요?
A: 1) 청약홈(www.applyhome.co.kr) 접속
2) 당첨자 조회 메뉴 클릭
3) 문자 알림 서비스 신청 가능
※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인
2-2

- 답변:
Q: 당첨자 발표는 어떻게 확인하나요?
A: 1) 청약홈(www.applyhome.co.kr) 접속
2) 당첨자 조회 메뉴 클릭
3) 문자 알림 서비스 신청 가능
※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인,
- 카테고리: 당첨/계약 
-점수: 1.0481367111206055 

- 답변:
Q: 청약홈 사이트는 어떻게 이용하나요?
A: 청약홈(www.applyhome.co.kr) - 한국부동산원 운영
1) 회원가입 후 공인인증서/간편인증 로그인
2) 청약 신청, 당첨 확인, 가점 계산 가능
3) 모바일 앱(청약홈)도 동일 서비스 제공,
- 카테고리: 기타 
-점수: 1.3872148990631104 

- 답변:
Q: 재당첨 제한이란 무엇인가요?
A: 당첨 후 일정 기간 다른 주택 청약 불가:
1) 투기과열지구: 10년
2) 청약과열지역: 7년
3) 수도권 공공주택: 5년
※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한),
- 카테고리: 당첨/계약 
-점수: 1.4651408195495605 

Loaded 벡터스토어 이용
2-1

k개 중 첫번째 답변 확인:
 Q: 당첨자 발표는 어떻게 확인하나요?
A: 1) 청약홈(www.applyhome.co.kr) 접속
2) 당첨자 조회 메뉴 클릭
3) 문자 알림 서비스 신청 가능
※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인


---
## 사이클 4: Retriever 구성

벡터 스토어에서 `as_retriever()`로 retriever를 만들고, `similarity` vs `mmr` 검색 타입과 `k=1,3,5` 결과를 비교하세요.

### ⭐[내용정리] 벡터스토어의 `as_retreiver' 및 리트리버

#### 1. 리트리버(Retriever)란?
- Retriever란 "쿼리(query) 하나를 받아서 관련 문서 리스트를 반환"하는 표준 인터페이스.
  - `query: str`  →  `[Retriever]`  →  `List[Document]`
  - `vectorstore.similarity_search()` 자체는 FAISS에 종속된 메서드이나, `retriever.invoke()`로 변환해서 사용. 기능은 사실상 같음.

#### 2. `as_retriever()` 인자 및 반환값 분석
```python
retriever = vectorstore.as_retriever(
    search_type="similarity",   # 검색 방식 선택
    search_kwargs={...}         # 검색 방식에 따른 세부 옵션
)
```
- 인자
  - `search_type`: 3가지 선택지 존재
    1. `"similarity"` (기본값): 유사도 점수 상위 k개를 반환
      ```python
      retriever = vectorstore.as_retriever(
          search_type="similarity",
          search_kwargs={"k": 3}
      )
      ```
    2. `"mmr"` (Maximal Marginal Relevance): 유사도가 높으면서 서로 중복되지 않는 k개를 반환. 결과들이 비슷한 내용으로 몰리는 현상을 방지.
      ```python
      retriever = vectorstore.as_retriever(
          search_type="mmr",
          search_kwargs={
              "k": 3,            # 최종 반환 개수
              "fetch_k": 20,     # 먼저 후보 20개를 뽑고, 그 중에서 다양성 고려해 3개 선택
              "lambda_mult": 0.5 # 0 = 다양성 최대, 1 = 유사도 최대
          }
      )
      ```
    3. `"similarity_score_threshold"`: 문턱값을 설정하여 그 이상인 문서만 반환. 관련
      - DB에서 나온 원래 점수가 '거리(낮을수록 좋음)' 방식이라면? 랭체인이 알아서 "유사도 점수(Relevance Score)"라는 0.0 ~ 1.0 사이의 점수로 뒤집어서 변환함.
      - 따라서, 여기서는 유사도 기준은 높을수록 유사한 값임.
        ```python
        retriever = vectorstore.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={
                "score_threshold": 0.7,  # 0.7 이상 유사도인 문서만 반환
                "k": 5                   # 최대 5개
            }
        )
        # 관련 문서가 없으면 빈 리스트 [] 반환
        ```
  - `search_kwargs` 전체 옵션 정리
    ```python
    search_kwargs = {
        "k":               3,                      # 반환 문서 수 (기본 4)
        "fetch_k":         20,                     # mmr 사용 시 후보 수
        "lambda_mult":     0.5,                    # mmr 사용 시 다양성 조절용 변수
        "score_threshold": 0.7,                    # 유사도 최소 기준
        "filter":          {"category": "청약통장"} # metadata 기준 필터링
    }
    ```

- 반환값
  - `as_retriever()`는 `VectorStoreRetriever` 객체를 반환
    ```python
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    print(type(retriever))
    # >>> <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>

    # ⭐.invoke()로 호출 → List[Document] 반환
    results = retriever.invoke("청약통장 1순위 조건")

    print(type(results))      
    # >>> <class 'list'>
    print(type(results[0]))   
    # >>> <class 'langchain_core.documents.base.Document'>
    print(len(results))       
    # >>> 3 (k=3 설정 시)

    # 결과 접근
    for doc in results:
        print(doc.page_content)
        print(doc.metadata)
    ```


#### 3. LangChain에서 Retriever 사용 방식
Retriever의 존재 이유는 "어떤 벡터 DB를 쓰든 체인 코드가 동일하게 작동하도록" 하는 추상화 레이어임. FAISS, Chroma, Pinecone 등 어떤 것이든 .as_retriever()로 변환하면 이후 rag_chain 코드는 수정하지 않아도 됨.

1. `.invoke()` 메서드 활용
  ```python
  retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

  docs = retriever.invoke("청약통장 가입 방법")
  # >>> [Document(...), Document(...), Document(...)]
  ```


2. LCEL 체인 (`|`파이프 연결) 활용
  - Retriever는 Runnable 인터페이스를 구현하므로 `|`로 연결 가능.
    ```python
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", "참고 문서:\n{context}\n\n문서 기반으로 답변해주세요."),
        ("user", "{question}")
    ])

    rag_chain = (
        {
            "context":  retriever | format_docs,   # ① 쿼리 → 문서 검색 → 텍스트 변환
            "question": RunnablePassthrough()       # ② 쿼리 그대로 통과
        }
        | rag_prompt         # ③ context + question → 프롬프트 구성
        | llm                # ④ LLM 호출
        | StrOutputParser()  # ⑤ AIMessage → str 변환
    )

    answer = rag_chain.invoke("청약통장 1순위 조건이 뭔가요?")

    ```
  - 데이터 흐름 시각화
    ```text
    rag_chain.invoke("청약통장 1순위 조건")
            │
            ├─→ retriever.invoke("청약통장 1순위 조건")
            │         ↓
            │   [Document(...), Document(...), Document(...)]
            │         ↓ format_docs()
            │   "Q: 1순위 조건은...\n\nQ: 청약통장이란..."
            │         ↓ (context에 담김)
            │
            ├─→ RunnablePassthrough()
            │   "청약통장 1순위 조건이 뭔가요?"
            │         ↓ (question에 담김)
            │
            → rag_prompt  →  llm  →  StrOutputParser()
                  ↓                         ↓
      "참고 문서: ...\n질문: ..."    "1순위 조건은 수도권 12개월..."
    ```




In [115]:
# 사이클 4: 벡터 스토어에서 as_retriever()로 retriever를 만들고,
# similarity vs mmr 검색 타입과 k=1,3,5 결과를 비교

# [계획]
# as_retriever()는 벡터스토어를 리트리버로 변환해주는 메서드
  # retriever = vectorstore.as_retriever() 사용해 변환
  # searh_type의 경우 similarity, mmr, similarity_score_threshold로 나뉨
""" retriever = vectorstore.as_retriever(
    search_type="similarity",   # 검색 방식 선택
    search_kwargs={...}         # 검색 방식에 따른 세부 옵션
)
"""
  # kwargs는 아래와 같음.
"""
search_kwargs = {
    "k":               3,                      # 반환 문서 수 (기본 4)
    "fetch_k":         20,                     # mmr 사용 시 후보 수
    "lambda_mult":     0.5,                    # mmr 사용 시 다양성 조절용 변수
    "score_threshold": 0.7,                    # 유사도 최소 기준
    "filter":          {"category": "청약통장"} # metadata 기준 필터링
}
"""
# ============================================================================
# 1) search_type = "similarity" 활용
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {'k':3}
)
print("1) search_type = \"similarity\" 활용")
print("---")
result1 = retriever.invoke(query)
for r in result1:
  print(r.page_content)
  print("---")

# 2) search_type = "mmr" 활용
print("\n\n")
print("2) search_type = \"mmr\" 활용")
print("---")
for k in range(1,6,2): # 1, 3, 5 비교용
  print(f"========== k값이 {k}인 경우 ==========")
  retriever2 = vectorstore.as_retriever(
      search_type = "mmr",
      search_kwargs = {
          'k':k,
          'fetch_k':10,
          "lambda_mult": 0.5,
          }
  )
  result2 = retriever2.invoke(query)
  for r2 in result2:
    print(r2.page_content)
    print("---")



1) search_type = "similarity" 활용
---
Q: 당첨자 발표는 어떻게 확인하나요?
A: 1) 청약홈(www.applyhome.co.kr) 접속
2) 당첨자 조회 메뉴 클릭
3) 문자 알림 서비스 신청 가능
※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인
---
Q: 청약홈 사이트는 어떻게 이용하나요?
A: 청약홈(www.applyhome.co.kr) - 한국부동산원 운영
1) 회원가입 후 공인인증서/간편인증 로그인
2) 청약 신청, 당첨 확인, 가점 계산 가능
3) 모바일 앱(청약홈)도 동일 서비스 제공
---
Q: 재당첨 제한이란 무엇인가요?
A: 당첨 후 일정 기간 다른 주택 청약 불가:
1) 투기과열지구: 10년
2) 청약과열지역: 7년
3) 수도권 공공주택: 5년
※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)
---



2) search_type = "mmr" 활용
---
========== k값이 1인 경우 ==========
Q: 당첨자 발표는 어떻게 확인하나요?
A: 1) 청약홈(www.applyhome.co.kr) 접속
2) 당첨자 조회 메뉴 클릭
3) 문자 알림 서비스 신청 가능
※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인
---
========== k값이 3인 경우 ==========
Q: 당첨자 발표는 어떻게 확인하나요?
A: 1) 청약홈(www.applyhome.co.kr) 접속
2) 당첨자 조회 메뉴 클릭
3) 문자 알림 서비스 신청 가능
※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인
---
Q: 가점제와 추첨제의 차이는 무엇인가요?
A: 가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)
추첨제: 무작위 추첨
1) 투기과열지구: 가점제 100%
2) 청약과열지역: 가점 75% + 추첨 25%
3) 기타: 가점 40% + 추첨 60%
---
Q: 청약통장 1순위 조건은 무엇인가요?
A: 1순위 조건은 주택 유형에 따라 다릅니다.
1) 민영주택: 수도권 12개월, 비수도권 

In [113]:
for i in range(1,6,2):
  print(i)

1
3
5


---
## 사이클 5: RAG 체인

`retriever | format_docs`를 context로 사용하는 RAG 체인을 LCEL로 만들고, 질문 5개로 테스트하세요. Weekend 1의 키워드 검색 대비 답변 품질 차이를 확인하세요.

---
### ⭐[내용정리] RAG 체인 및 프롬프트 작성 복습

#### 1. `format_docs` 함수?
- Document 객체의 리스트의 page_content를 꺼내어 문자열로 변환.
- Retriever는 `List[Document]`를 반환하므로, 프롬프트(prompt)의 `{context}` 자리에는 str이 들어가야함. 따라서 변환해주는 함수가 필요
  ```python
  # 인자: List[Document]
  # 반환: str

  def format_docs(docs):
      # docs 안의 각 Document.page_content를 꺼내서 하나의 문자열로 합침
      return "\n\n".join(doc.page_content for doc in docs)

  # 예시 결과
  # "Q: 청약통장 1순위 조건은?\nA: 수도권 12개월...\n\nQ: 주택청약이란?\nA: 만능통장..."
  ```

#### 2. RAG용 프롬프트 `ChatPromptTemplate.from_messages()`
- `{context}` 플레이스 홀더를 추가하여 해당 위치에 retriever로 반환된 값을 넣음.
- 관련 라이브러리는 `langchain_core.prompt`
  ```python
  from langchain_core.prompts import ChatPromptTemplate

  # 기본 구조 (튜플 리스트)
  prompt = ChatPromptTemplate.from_messages([
      ("system", "...{context}..."),   # context 플레이스홀더
      ("user",   "{question}")         # question 플레이스홀더
  ])

  # 나중에 invoke 시 딕셔너리로 값을 채워줌
  # prompt.invoke({"context": "...", "question": "..."})

  ```
#### 3. `Runnable` 인터페이스 개념 (ChatPromptTemplate 및 RunnablePassthrough)
- ChatPromptTemplate — 프롬프트 내용 담당
  - ChatPromptTemplate은 Runnable을 상속하지만, 목적은 LLM에 전달할 메시지를 구조화된 템플릿으로 만드는 것.
  - {변수명} 플레이스홀더에 실제 값을 채워 넣어 완성된 메시지를 작성.
- RunnablePassthrough — 데이터 흐름 담당
  - LCEL 체인에서 입력값을 아무 변환 없이 그대로 다음 단계로 통과
  - RAG 체인의 딕셔너리 분기 구조에서 핵심 역할

-  Runnable 인터페이스 개념
  - LCEL의 | 파이프에 연결할 수 있는 모든 것은 Runnable 인터페이스를 구현해야 함.
  Runnable은 "파이프에 꽂을 수 있는 표준 소켓"이고, 그 소켓을 구현한 것들이 다음과 같음.
  | 구현체                  | 역할           | invoke() 입력 → 출력            |
| -------------------- | ------------ | --------------------------- |
| ChatPromptTemplate   | 메시지 템플릿 구성   | dict → ChatPromptValue      |
| ChatOpenAI (llm)     | LLM API 호출   | ChatPromptValue → AIMessage |
| StrOutputParser      | 응답 텍스트 추출    | AIMessage → str             |
| RunnablePassthrough  | 데이터 그대로 통과   | X → X                       |
| retriever            | 유사 문서 검색     | str → List[Document]        |
| format_docs (커스텀 함수) | 문서 리스트 → 문자열 | List[Document] → str        |

#### - 지난주(Weekend 1)의 방식과 비교
- 지난주
  - 문서 검색 방법: LLM이 키워드 추출 -> keywords 필드 매칭
  - context 구성: `lambda x: make_context(x, FAQ_DATA)`
  - 키워드 불일치 시 검색 실패
- 지금 (RAG 체인) 방식 이용 시
  - 문서 검색 방법: 드 매칭	FAISS 벡터 유사도 검색
  - context 구성: `retriever | format_docs`
  - 의미 기반 검색

In [120]:
# 사이클 5
# 1) retriever | format_docs를 context로 사용하는 RAG 체인을 LCEL로 만들고, 질문 5개로 테스트하세요.
# 2) Weekend 1의 키워드 검색 대비 답변 품질 차이를 확인하세요.

from langchain_core.runnables import RunnablePassthrough
"""
ChatPromptTemplate  →  "LLM에 보낼 메시지 내용을 어떻게 구성할까?"  (내용/포맷 담당)
RunnablePassthrough →  "데이터를 체인 어느 곳에 어떻게 흘려보낼까?"  (흐름/배관 담당)
"""

# [계획]
# 1)
#우선은 retriever 구성, retriever는 어떻게 구성했었지?
  # (1) 벡터스토어 초기화
    # vectorstore = FAISS.from_documents(docs, embeddings)
  # (2) 벡터스토어를 리트리버로 변환
    # vectorstore.as_retriever(search_type = "similarity", search_kwargs = {'k':3})
    # search_type은 그냥, mmr, threshold 세가지?
  # (3) 리트리버에 쿼리(query)를 넣은 후 invoke()했음 -> 관련 문서 반환됨
    # vectorstore.invoke(query)
# 그리고, format_docs 함수 가져와야함 -> 리트리버 반환형식은 Document 객체이므로 문자열로 변환해야함.

# 2) RAG 체인 만들어야함.
  # (1) ChatPromptTemplate 사용해서 prompt 템플릿 만들어야함.
    # System..과 Human.. 부분이 있었고 {context} 및 {query}등 플레이스홀더 존재.
    # {context} 플레이스 홀더에 리트리버로 반환된 문자열을 넣어줌.
  # (2) 체인 구성할 때 RunnablePassthrough 이용해야함, query를 우선 통과시키고 리트리버로 먼저 제공
    # 그 후 `|` 등으로 prompt | llm | parser 이어주고 끝.

# ===================================================================

# 1) 리트리버 구성

# 벡터스토어 초기화
vectorstore = FAISS.from_documents(docs, embeddings) # docs는 맨 위에서 for루프로 텍스트 추출 -> document 객체 만듦
# 벡터스토어를 리트리버로 변환
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.8,  # 0.7 이상 유사도인 문서만 반환
        "k": 3                   # 최대 3개
    }
)
# 사용 시 리트리버에 쿼리(query)를 넣은 후 invoke()
# format_docs 함수
def format_docs(docs):
    # docs 안의 각 Document.page_content를 꺼내서 하나의 문자열로 합침
    return "\n\n".join(doc.page_content for doc in docs)

# 2) RAG 체인 구성

# 프롬프트 구성: ChatPromptTemplate
  # from langchain_core.prompts import ChatPromptTemplate
  # ChatPromptTemplate.from_messages
    # a. ⭐튜플 방식 + 플레이스홀더({변수명})을 가장 많이 사용
      # role: "assistant": AI의 이전 응답 관련
    # b. 메시지 클래스(SystemMessage,HumanMessage,AIMessage)도 사용 가능하긴 한데, 커스터마이징 불가..
prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자의 질문에 {context}를 상세하게 참고하여 답변해주세요."),
    ("user", "{query}")
])

# 체인 구성
rag_chain = (
    {
     "context": retriever | format_docs,
     "query": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 질문 5개
SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
]

for i in range(5):
  q = SAMPLE_TEST_QUERIES[i].get('query')
  return_message = rag_chain.invoke(q)
  print(f"{i}번째 질문 응답\n{return_message}")
  print("="*20)


0번째 질문 응답
청약통장에 가입하려면 다음과 같은 절차를 따르면 됩니다:

1. **은행 선택**: 청약통장은 주택청약종합저축, 청약저축 등 여러 종류가 있으며, 이를 취급하는 은행이나 금융기관을 선택해야 합니다. 주요 은행으로는 국민은행, 신한은행, 우리은행, 하나은행 등이 있습니다.

2. **신분증 준비**: 가입 시 본인 확인을 위해 신분증(주민등록증, 운전면허증 등)을 준비해야 합니다.

3. **방문 또는 온라인 신청**: 선택한 은행의 지점을 방문하거나, 해당 은행의 인터넷 뱅킹 또는 모바일 앱을 통해 온라인으로 신청할 수 있습니다. 온라인 신청 시에는 본인 인증 절차가 필요할 수 있습니다.

4. **가입 신청서 작성**: 청약통장 가입 신청서를 작성합니다. 이때, 개인 정보와 청약통장 종류를 선택해야 합니다.

5. **가입금액 결정**: 청약통장에 가입할 때 초기 납입금액을 결정해야 합니다. 일반적으로 최소 가입금액이 정해져 있으니, 이를 확인하고 납입합니다.

6. **통장 발급**: 모든 절차가 완료되면 청약통장이 발급됩니다. 통장 발급 후에는 정기적으로 납입을 하여 청약 자격을 유지해야 합니다.

7. **청약 신청**: 청약통장에 일정 금액 이상이 적립되면, 주택 청약을 신청할 수 있는 자격이 주어집니다.

가입 후에는 청약통장의 적립금과 이자, 청약 자격 요건 등을 잘 관리하는 것이 중요합니다. 추가적인 정보나 궁금한 점이 있다면 해당 은행의 고객센터에 문의하는 것도 좋은 방법입니다.


/usr/local/lib/python3.12/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='FAQ004', metadata={'category': '청약통장'}, page_content='Q: 청약통장 1순위 조건은 무엇인가요?\nA: 1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입'), np.float32(0.0850929)), (Document(id='FAQ013', metadata={'category': '일반공급'}, page_content='Q: 가점제와 추첨제의 차이는 무엇인가요?\nA: 가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%'), np.float32(0.041134417)), (Document(id='FAQ006', metadata={'category': '청약자격'}, page_content='Q: 무주택자 기준은 무엇인가요?\nA: 본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함'), np.float32(-0.05818057))]
  self.vectorstore.similarity_search_with_relevance_scores(


1번째 질문 응답
1순위가 되기 위해서는 여러 가지 요소가 필요합니다. 여기서는 일반적인 상황에서 1순위를 차지하기 위한 몇 가지 중요한 요소를 설명하겠습니다.

1. **목표 설정**: 명확하고 구체적인 목표를 설정하는 것이 중요합니다. 무엇을 1순위로 두고 싶은지 명확히 해야 합니다.

2. **계획 수립**: 목표를 달성하기 위한 구체적인 계획을 세워야 합니다. 단계별로 필요한 행동을 정리하고, 일정도 설정하는 것이 좋습니다.

3. **노력과 헌신**: 목표를 이루기 위해서는 지속적인 노력과 헌신이 필요합니다. 꾸준히 노력하고, 실패를 두려워하지 않는 자세가 중요합니다.

4. **자기 관리**: 시간 관리, 스트레스 관리 등 자기 관리를 잘 해야 합니다. 효율적으로 시간을 사용하고, 건강을 유지하는 것이 목표 달성에 큰 도움이 됩니다.

5. **네트워킹**: 관련 분야의 사람들과의 관계를 구축하는 것도 중요합니다. 멘토나 동료와의 교류를 통해 더 많은 정보를 얻고, 기회를 만들 수 있습니다.

6. **피드백 수용**: 자신의 행동이나 결과에 대한 피드백을 받아들이고, 이를 바탕으로 개선하는 자세가 필요합니다.

7. **지속적인 학습**: 변화하는 환경에 적응하기 위해 지속적으로 배우고 성장하는 것이 중요합니다. 새로운 기술이나 지식을 습득하는 것이 도움이 됩니다.

이러한 요소들을 종합적으로 고려하고 실천한다면, 원하는 분야에서 1순위를 차지하는 데 큰 도움이 될 것입니다.


2번째 질문 응답
신혼부부 특별공급(특공)은 주택청약제도 중 하나로, 신혼부부가 주택을 구매할 때 우선적으로 공급받을 수 있는 제도입니다. 신혼부부 특공의 자격 요건은 다음과 같습니다:

1. **혼인 기간**: 일반적으로 혼인신고를 한 지 7년 이내의 신혼부부가 대상입니다. 다만, 자녀가 있는 경우에는 혼인 기간에 대한 제한이 완화될 수 있습니다.

2. **소득 기준**: 신혼부부의 소득이 일정 기준 이하이어야 합니다. 이 기준은 지역에 따라 다를 수 있으며, 보통 중위소득의 100% 이하로 설정됩니다.

3. **주택 소유 여부**: 신청자는 본인 명의의 주택이 없어야 하며, 배우자와 함께 주택을 소유하지 않아야 합니다.

4. **청약통장**: 청약통장이 있어야 하며, 일정 기간 이상 가입되어 있어야 합니다. 보통 1년 이상 가입된 청약통장이 필요합니다.

5. **거주 요건**: 해당 주택이 위치한 지역에 일정 기간 거주해야 할 수 있습니다.

신혼부부 특공의 구체적인 자격 요건은 지역 및 공급되는 주택의 종류에 따라 다를 수 있으므로, 해당 지역의 주택공사나 관련 기관의 공식 웹사이트를 통해 최신 정보를 확인하는 것이 좋습니다.


/usr/local/lib/python3.12/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='FAQ013', metadata={'category': '일반공급'}, page_content='Q: 가점제와 추첨제의 차이는 무엇인가요?\nA: 가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%'), np.float32(-0.03637135)), (Document(id='FAQ006', metadata={'category': '청약자격'}, page_content='Q: 무주택자 기준은 무엇인가요?\nA: 본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함'), np.float32(-0.21148634)), (Document(id='FAQ004', metadata={'category': '청약통장'}, page_content='Q: 청약통장 1순위 조건은 무엇인가요?\nA: 1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입'), np.float32(-0.21932352))]
  self.vectorstore.similarity_search_with_relevance_scores(


3번째 질문 응답
가점이 높으면 일반적으로 유리한 경우가 많습니다. 가점은 특정한 기준이나 조건에 따라 부여되는 점수로, 주로 경쟁이 있는 상황에서 우선권이나 혜택을 받을 수 있는 기회를 제공합니다. 예를 들어, 취업, 대학 입학, 주택 청약 등 다양한 분야에서 가점이 높은 경우 더 유리한 위치에 놓일 수 있습니다.

가점이 높다는 것은 해당 기준에서 더 많은 자격이나 조건을 충족했음을 의미하므로, 경쟁자들 사이에서 우위를 점할 수 있는 요소가 됩니다. 그러나 가점의 유리함은 상황에 따라 다를 수 있으며, 가점 외에도 다른 요소들이 중요할 수 있습니다. 따라서 가점이 높다고 해서 무조건 성공하는 것은 아니지만, 일반적으로는 긍정적인 영향을 미치는 경우가 많습니다.


/usr/local/lib/python3.12/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='FAQ017', metadata={'category': '당첨/계약'}, page_content='Q: 당첨자 발표는 어떻게 확인하나요?\nA: 1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인'), np.float32(0.25882602)), (Document(id='FAQ023', metadata={'category': '기타'}, page_content='Q: 청약홈 사이트는 어떻게 이용하나요?\nA: 청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공'), np.float32(0.019927979)), (Document(id='FAQ020', metadata={'category': '당첨/계약'}, page_content='Q: 재당첨 제한이란 무엇인가요?\nA: 당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)'), np.float32(-0.03536129))]
  self.vectorstore.similarity_search_with_relevance_scores(


4번째 질문 응답
당첨 여부를 확인하는 방법은 주로 다음과 같습니다:

1. **복권 판매점 방문**: 구매한 복권의 종류에 따라, 가까운 복권 판매점에 가서 직접 확인할 수 있습니다. 판매점 직원이 당첨 여부를 확인해 줄 수 있습니다.

2. **온라인 확인**: 많은 복권은 공식 웹사이트나 모바일 앱을 통해 당첨 여부를 확인할 수 있는 서비스를 제공합니다. 구매한 복권의 번호를 입력하면 쉽게 확인할 수 있습니다.

3. **전화 확인**: 일부 복권의 경우, 고객 서비스 센터에 전화하여 당첨 여부를 확인할 수 있는 방법도 있습니다.

4. **SNS 및 뉴스**: 당첨 번호가 발표된 후, 뉴스나 소셜 미디어를 통해 확인할 수도 있습니다. 공식 발표를 통해 확인하는 것이 가장 정확합니다.

당첨된 경우, 상금 수령 방법은 복권의 종류와 금액에 따라 다를 수 있으니, 해당 규정을 잘 확인하는 것이 중요합니다.


---
## 사이클 6: 검색 결과 검증

`SAMPLE_TEST_QUERIES` 5개로 RAG 체인이 올바른 FAQ를 찾는지 검증하세요. `similarity_search_with_score()`로 유사도 점수도 함께 확인하고, 기대한 FAQ ID가 검색 결과에 포함되는지 O/X로 판정하세요. 결과를 표로 정리하세요.

In [121]:
# 사이클 6
  # SAMPLE_TEST_QUERIES 5개로 RAG 체인이 올바른 FAQ를 찾는지 검증
  # similarity_search_with_score()로 유사도 점수도 함께 확인
  # 기대한 FAQ ID가 검색 결과에 포함되는지 O/X로 판정


# [계획]
  # 1) vectorstore의 similarity_search_with_score(query, k=3)를 사용
    # 반환: List[Tuple[Document, float]]
    # float는 L2 거리이므로 낮을수록 유사
  # 2) 5개 테스트 쿼리에 대해 반복
    # 각 쿼리의 expected_faq_id가 검색 결과의 doc.id 리스트에 있는지 확인
    # ⭐ SAMPLE_TEST_QUERIES에 expected_faq_id 있음
    # O/X 판정
  # 3) 결과를 표(table) 형태로 출력
    # 쿼리 | 기대 FAQ ID | 검색 1위 ID | 1위 점수 | O/X

# ===================================================================

# 벡터스토어 생성
vectorstore = FAISS.from_documents(docs, embeddings)

print(f"{'쿼리':<25} | {'기대ID':<8} | {'1위 ID':<8} | {'1위 점수':<10} | {'판정'}")
print("-" * 75)

correct_count = 0

for tq in SAMPLE_TEST_QUERIES:
    q = tq.get('query')
    expected_id = tq.get('expected_faq_id')

    # similarity_search_with_score: L2 거리 기반, 낮을수록 유사
    results = vectorstore.similarity_search_with_score(q, k=3)

    # 검색된 문서들의 id 리스트 추출
    found_ids = [doc.id for doc, score in results]

    # 1위 문서 정보
    top_doc, top_score = results[0]
    top_id = top_doc.id

    # O/X 판정: k개 안에 expected_id가 있으면 O
    verdict = "O" if expected_id in found_ids else "X"
    if verdict == "O":
        correct_count += 1

    print(f"{q:<25} | {expected_id:<8} | {top_id:<8} | {top_score:<10.4f} | {verdict}")

print("-" * 75)
print(f"정확도 (k=3 기준): {correct_count}/{len(SAMPLE_TEST_QUERIES)} ({correct_count/len(SAMPLE_TEST_QUERIES)*100:.0f}%)")



쿼리                        | 기대ID     | 1위 ID    | 1위 점수      | 판정
---------------------------------------------------------------------------
청약통장 가입하려면 어떻게 해요?        | FAQ001   | FAQ023   | 0.9954     | O
1순위 되려면 뭐가 필요해요?          | FAQ004   | FAQ004   | 1.2964     | O
신혼부부 특공 자격이 궁금해요          | FAQ010   | FAQ010   | 1.0097     | O
가점이 높으면 유리한가요?            | FAQ013   | FAQ013   | 1.4660     | O
당첨되면 어떻게 확인해요?            | FAQ017   | FAQ017   | 1.0481     | O
---------------------------------------------------------------------------
정확도 (k=3 기준): 5/5 (100%)


---
## 사이클 7: 소스 문서 표시

RAG 답변에 참고한 FAQ 출처(ID, 카테고리)를 함께 반환하는 체인을 만드세요. `RunnableParallel`로 answer와 sources를 동시에 가져오세요.

---
### ⭐[내용정리] RunnableParallel과 소스 문서 추적

#### 1. RunnableParallel이란?
- 여러 Runnable을 병렬로 실행하여, 각각의 결과를 딕셔너리로 묶어 반환하는 유틸리티.
- `from langchain_core.runnables import RunnableParallel`
- 기본 구조
  ```python
  from langchain_core.runnables import RunnableParallel

  parallel = RunnableParallel(
      answer=chain_a,   # Runnable A
      sources=chain_b   # Runnable B
  )
  result = parallel.invoke("입력")
  # >>> {"answer": chain_a 결과, "sources": chain_b 결과}
  ```
- 사이클 5에서 사용한 딕셔너리 분기 구조 `{"context": ..., "query": ...}`도 내부적으로 `RunnableParallel`과 동일하게 동작.
  ```python
  # 아래 두 가지는 동일
  {"context": retriever | format_docs, "query": RunnablePassthrough()}
  RunnableParallel(context=retriever | format_docs, query=RunnablePassthrough())
  ```

#### 2. 소스 문서 추적
- 기존 RAG 체인은 `retriever | format_docs`로 문서를 문자열로 변환하므로, 원본 Document 객체(id, metadata)가 사라짐.
- retriever 결과를 두 곳에서 동시에 사용.
  1. `answer` 브랜치: retriever → format_docs → prompt → llm → parser (기존과 동일)
  2. `sources` 브랜치: retriever → 각 Document의 id, category 추출
- 이를 위해 retriever를 먼저 실행한 결과를 공유하거나, `RunnableParallel`로 분기.

#### 3. 구현 방식
- 방법 A: `RunnablePassthrough.assign()` 사용 — 기존 데이터에 새로운 키를 추가
  ```python
  # assign()은 기존 입력 딕셔너리에 새 키를 추가하는 메서드
  chain = RunnablePassthrough.assign(
      context=lambda x: format_docs(retriever.invoke(x["query"])),
      sources=lambda x: retriever.invoke(x["query"])
  )
  ```
- 방법 B: `RunnableParallel`로 answer 체인과 sources 체인을 분리
  ```python
  # retriever를 호출하여 문서를 가져온 후
  # answer 쪽에는 format_docs를 적용, sources 쪽에는 Document 원본 유지
  
  def extract_sources(docs):
      return [{"id": doc.id, "category": doc.metadata.get("category")} for doc in docs]
  
  rag_with_sources = RunnableParallel(
      answer=rag_chain,             # 기존 RAG 체인 (str 반환)
      sources=retriever | extract_sources  # 소스 정보 (list[dict] 반환)
  )
  ```


In [122]:
# 사이클 7
  # RAG 답변에 참고한 FAQ 출처(ID, 카테고리)를 함께 반환하는 체인
  # RunnableParallel로 answer와 sources를 동시에 가져옴

from langchain_core.runnables import RunnableParallel

# [계획]
  # 1) retriever는 이미 있음. (vectorstore.as_retriever())
  # 2) 참고한 FAQ 출처는 retriever로 문서를 가져온 뒤, 각 Document의 id와 category를 추출
    # retriever.invoke(query) → List[Document]
    # 각 doc에서 doc.id, doc.metadata.get('category')를 딕셔너리로 만듦
  # 3) RunnableParallel로 두 체인을 병렬 실행

# ===================================================================


# 벡터스토어 & 리트리버 구성 (threshold 없이 similarity 사용)
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# format_docs: Document 리스트 -> 문자열 (RAG context용)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# extract_sources: Document 리스트 -> 출처 정보 리스트
def extract_sources(docs):
    return [{"id": doc.id, "category": doc.metadata.get("category")} for doc in docs]

# 프롬프트 구성
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주택청약 FAQ 챗봇입니다. 아래 참고 문서를 기반으로 답변해주세요.\n\n참고 문서:\n{context}"),
    ("user", "{query}")
])


# 체인 구성
rag_chain = (
    {
        "context": retriever | format_docs,
        "query": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# RunnableParallel로 answer와 sources를 동시에 가져옴
rag_with_sources = RunnableParallel(
    answer=rag_chain,
    sources=retriever | extract_sources
)

# 테스트
test_query = "신혼부부 특공 자격이 궁금해요"
result = rag_with_sources.invoke(test_query)

print("답변:")
print(result["answer"])
print("\n 참고 FAQ 출처:")
for src in result["sources"]:
    print(f"  - {src['id']} ({src['category']})")



답변:
신혼부부 특별공급의 자격 조건은 다음과 같습니다:

1) 혼인기간 7년 이내의 무주택 세대주
2) 소득: 도시근로자 월평균소득의 100%에서 140% 사이
3) 전용면적 85㎡ 이하의 주택
4) 혼인기간이 짧을수록, 자녀가 많을수록 가점이 높습니다.
5) 예비 신혼부부도 신청이 가능합니다.

이 조건들을 충족하면 신혼부부 특별공급에 신청할 수 있습니다.

 참고 FAQ 출처:
  - FAQ010 (특별공급)
  - FAQ005 (청약자격)
  - FAQ009 (특별공급)


---
## 사이클 8: FAQChatbotV2 클래스

vectorstore, retriever, rag_chain을 하나로 묶는 `FAQChatbotV2` 클래스를 만드세요. `ask(question)` 메서드가 answer, sources, time을 반환하도록 하세요.

In [126]:
# 사이클 8
  # vectorstore, retriever, rag_chain을 하나로 묶는 FAQChatbotV2 클래스
  # ask(question) 메서드가 answer, sources, time을 반환
import time

# [계획]
  # 1) __init__  초기화
    # faq_data를 받아서 Document 리스트로 변환
    # FAISS.from_documents()로 벡터스토어 생성
    # as_retriever()로 리트리버 생성
    # RunnableParallel로 answer + sources 체인 구성
  # 2) ask(question) 메서드
    # time.time()으로 시작 시간 기록
    # rag_with_sources.invoke(question)으로 답변 + 출처 가져옴
    # 응답 시간 계산
    # dict로 반환 {"answer": ..., "sources": ..., "time": ...}

# ==============================================================


class FAQChatbotV2:
    def __init__(self, faq_data, model_name="gpt-4o-mini"):
        # LLM 및 임베딩 모델 초기화
        self.llm = ChatOpenAI(model=model_name, temperature=0)
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        # FAQ 데이터를 Document 리스트로 변환 (사이클 1 로직)
        self.docs = []
        for sample in faq_data:
            q = f"Q: {sample.get('question')}\n"
            a = f"A: {sample.get('answer')}"
            content = q + a
            faq_id = sample.get('id')
            cat = sample.get('category')

            self.docs.append(Document(
                page_content=content,
                id=faq_id,
                metadata={"category": cat}
            ))

        # FAISS 벡터스토어 생성
        self.vectorstore = FAISS.from_documents(self.docs, self.embeddings)

        # 리트리버 구성
        self.retriever = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 3}
        )
        # 프롬프트
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "당신은 주택청약 FAQ 챗봇입니다. 아래 참고 문서를 기반으로 정확하게 답변해주세요.\n\n참고 문서:\n{context}"),
            ("user", "{query}")
        ])

        # RAG 체인
        self.rag_chain = (
            {
                "context": self.retriever | self._format_docs,
                "query": RunnablePassthrough()
            }
            | self.prompt
            | self.llm
            | StrOutputParser()
        )

        # answer + sources 병렬 체인
        self.chain_with_sources = RunnableParallel(
            answer=self.rag_chain,
            sources=self.retriever | self._extract_sources
        )

        print(f"FAQChatbotV2 초기화 완료 (문서 {len(self.docs)}개, 모델: {model_name})")

    def _format_docs(self, docs):
        """Document 리스트 -> 문자열 변환 (RAG context용)"""
        return "\n\n".join(doc.page_content for doc in docs)

    def _extract_sources(self, docs):
        """Document 리스트 -> 출처 정보 리스트 추출"""
        return [{"id": doc.id, "category": doc.metadata.get("category")} for doc in docs]

    def ask(self, question: str):
        """질문을 받아 answer, sources, time을 반환"""
        start = time.time()
        result = self.chain_with_sources.invoke(question)
        elapsed = time.time() - start

        return {
            "answer": result["answer"],
            "sources": result["sources"],
            "time": round(elapsed, 2)
        }

In [127]:
chatbot = FAQChatbotV2(SAMPLE_FAQ_DATA)

result = chatbot.ask("1순위 되려면 뭐가 필요해요?")

print(f"답변:\n{result['answer']}")
print(f"\n출처:")
for src in result["sources"]:
    print(f"  - {src['id']} ({src['category']})")
print(f"\n응답 시간: {result['time']}초")


FAQChatbotV2 초기화 완료 (문서 10개, 모델: gpt-4o-mini)
답변:
1순위 조건은 주택 유형에 따라 다릅니다.

1) 민영주택: 수도권에서 12개월, 비수도권에서 6개월 이상 청약통장을 유지해야 하며, 예치금도 필요합니다.
2) 국민주택: 수도권에서 12개월(24회), 비수도권에서 6개월(12회) 이상 청약통장을 유지해야 합니다.
3) 투기과열지구: 2년 이상 청약통장을 유지하고, 24회 납입해야 합니다.

출처:
  - FAQ004 (청약통장)
  - FAQ013 (일반공급)
  - FAQ006 (청약자격)

응답 시간: 3.85초


---
## 사이클 9: Gradio UI v2

`gr.ChatInterface`로 RAG 기반 FAQ 챗봇 UI를 만드세요. 답변에 참고 FAQ 출처도 포함해서 표시하세요.

In [128]:
# 사이클 9
import gradio as gr

def respond(message, history):
    """Gradio ChatInterface 콜백 함수"""
    result = chatbot.ask(message)
    answer = result["answer"]
    sources = result["sources"]
    time_taken = result["time"]

    # 출처 텍스트 구성
    source_text = "\n".join([f"- {s['id']} ({s['category']})" for s in sources])
    full_response = f"{answer}\n\n---\n📎 참고 FAQ:\n{source_text}\n⏱️ 응답 시간: {time_taken}초"
    return full_response

# Gradio ChatInterface 구성
demo = gr.ChatInterface(
    fn=respond,
    title="🏠 주택청약 FAQ 챗봇 V2 (RAG)",
    description="주택청약 관련 질문을 입력하세요. FAISS 벡터 검색 기반으로 관련 FAQ를 찾아 답변합니다.",
    type="messages",
    examples=[
        "청약통장 가입하려면 어떻게 해요?",
        "1순위 되려면 뭐가 필요해요?",
        "신혼부부 특공 자격이 궁금해요",
        "가점이 높으면 유리한가요?",
        "당첨되면 어떻게 확인해요?",
    ],
)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://813bd5921ad48c75ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 사이클 10: 통합 테스트

전체 RAG 파이프라인을 10개 질문으로 테스트하세요. 각 질문의 응답 시간, 검색된 카테고리, 답변 길이를 포함한 결과표를 출력하고, 전체 정확도와 평균 응답 시간을 요약하세요.

In [129]:
# 사이클 10
import time


# 10개 테스트 질문 (기존 5개 + 추가 5개)
TEST_QUERIES_10 = [
    # 기존 5개
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
    # 추가 5개
    {"query": "무주택자 기준이 뭔가요?", "expected_category": "청약자격", "expected_faq_id": "FAQ006"},
    {"query": "특별공급 종류 알려주세요", "expected_category": "특별공급", "expected_faq_id": "FAQ009"},
    {"query": "재당첨 제한 기간이 얼마나 되나요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ020"},
    {"query": "청약홈 사이트 어떻게 쓰나요?", "expected_category": "기타", "expected_faq_id": "FAQ023"},
    {"query": "청약 자격 조건 알려주세요", "expected_category": "청약자격", "expected_faq_id": "FAQ005"},
]

# chatbot 인스턴스 생성 (또는 재사용)
chatbot = FAQChatbotV2(SAMPLE_FAQ_DATA)

# 결과 수집
results_table = []

print(f"{'#':<3} | {'쿼리':<25} | {'기대ID':<8} | {'검색 카테고리':<15} | {'답변길이':<6} | {'시간(초)':<8} | {'판정'}")
print("-" * 95)

correct_count = 0
total_time = 0

for i, tq in enumerate(TEST_QUERIES_10):
    q = tq.get('query')
    expected_id = tq.get('expected_faq_id')

    # FAQChatbotV2.ask() 호출
    result = chatbot.ask(q)

    answer = result["answer"]
    sources = result["sources"]
    elapsed = result["time"]
    total_time += elapsed

    # 검색된 카테고리 (상위 문서들의 카테고리)
    searched_categories = ", ".join([s["category"] for s in sources])

    # 검색된 FAQ ID 리스트
    found_ids = [s["id"] for s in sources]

    # O/X 판정
    verdict = "O" if expected_id in found_ids else "X"
    if verdict == "O":
        correct_count += 1

    print(f"{i+1:<3} | {q:<25} | {expected_id:<8} | {searched_categories:<15} | {len(answer):<6} | {elapsed:<8.2f} | {verdict}")

    results_table.append({
        "query": q,
        "expected_id": expected_id,
        "found_ids": found_ids,
        "categories": searched_categories,
        "answer_len": len(answer),
        "time": elapsed,
        "verdict": verdict
    })

# 요약
print("=" * 95)
print(f"\n📊 통합 테스트 요약")
print(f"  - 전체 정확도 (k=3 기준): {correct_count}/{len(TEST_QUERIES_10)} ({correct_count/len(TEST_QUERIES_10)*100:.0f}%)")
print(f"  - 평균 응답 시간: {total_time/len(TEST_QUERIES_10):.2f}초")
print(f"  - 총 소요 시간: {total_time:.2f}초")
print(f"  - 평균 답변 길이: {sum(r['answer_len'] for r in results_table)/len(results_table):.0f}자")


FAQChatbotV2 초기화 완료 (문서 10개, 모델: gpt-4o-mini)
#   | 쿼리                        | 기대ID     | 검색 카테고리         | 답변길이   | 시간(초)    | 판정
-----------------------------------------------------------------------------------------------
1   | 청약통장 가입하려면 어떻게 해요?        | FAQ001   | 기타, 당첨/계약, 청약통장 | 192    | 4.42     | O
2   | 1순위 되려면 뭐가 필요해요?          | FAQ004   | 청약통장, 일반공급, 청약자격 | 190    | 4.09     | O
3   | 신혼부부 특공 자격이 궁금해요          | FAQ010   | 특별공급, 청약자격, 특별공급 | 148    | 2.93     | O
4   | 가점이 높으면 유리한가요?            | FAQ013   | 일반공급, 청약자격, 청약통장 | 138    | 2.26     | O
5   | 당첨되면 어떻게 확인해요?            | FAQ017   | 당첨/계약, 기타, 당첨/계약 | 138    | 2.09     | O
6   | 무주택자 기준이 뭔가요?             | FAQ006   | 청약자격, 청약자격, 특별공급 | 154    | 2.83     | O
7   | 특별공급 종류 알려주세요             | FAQ009   | 특별공급, 특별공급, 당첨/계약 | 156    | 3.31     | O
8   | 재당첨 제한 기간이 얼마나 되나요?       | FAQ020   | 당첨/계약, 청약통장, 당첨/계약 | 114    | 2.46     | O
9   | 청약홈 사이트 어떻게 쓰나요?          | FAQ023   | 기타, 당첨/계약, 청약통장 | 178    | 1.85     |

---
## Weekend 2 완료! 🎉

다음 주: 프롬프트 엔지니어링 + 메모리 + 최종 완성